# Logistic regression: predicting yes or no

Data Scientist track, lesson DS4. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** fit a logistic regression, read its coefficients as odds ratios on one scale with an interval, check that its chances are calibrated, choose a threshold from error costs on a validation part, and report precision and recall on the test part.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The splits

The columns and the split of lesson DS2 (the same test part), and a validation part cut from the training part. The test part is touched once, in unit 4. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)
print(len(X_fit), len(X_val), len(X_test))

## Unit 1: A line for the odds

**Predict first.** Each extra active day multiplies the odds of churning by a factor. Is it about 0.2, about 0.6 or about 1.0?

In [ ]:
one = LogisticRegression(max_iter=1000).fit(X_fit[["active_days_7d"]], y_fit)       # one column: active days in the first week
w, w0 = float(one.coef_[0][0]), float(one.intercept_[0])
print(round(w, 3), round(float(np.exp(w)), 3))
days = pd.DataFrame({"active_days_7d": [0, 1, 2, 3]})
chance = one.predict_proba(days)[:, 1]
print(np.round(chance, 3))
print(np.round(chance[1:] / (1 - chance[1:]) / (chance[:-1] / (1 - chance[:-1])), 3))       # odds at 1, 2, 3 days over the odds a day earlier
print(np.round(y_fit.groupby(X_fit["active_days_7d"]).mean().loc[[0, 1, 2, 3]].to_numpy(), 3))

## Unit 2: Coefficients on one scale, with an interval

**Predict first.** In unit 1 one extra active day gave an odds ratio of 0.62. Now `active_days_7d` sits in a model with the other week-1 columns, per standard deviation (about one day). Will its odds ratio be about the same, much closer to 1, or below 0.3?

In [ ]:
def recipe(num):
    """One-hot channel, platform and region (one level left out of each), the number columns scaled, then the model."""
    cat = OneHotEncoder(drop=["organic_search", "web", "north_america"])
    return make_pipeline(ColumnTransformer([("cat", cat, STATIC), ("num", StandardScaler(), num)]), LogisticRegression(max_iter=1000))


def odds_ratios(m):
    return pd.Series(np.exp(m[-1].coef_[0]), index=[n.split("__")[1] for n in m[0].get_feature_names_out()])


full = recipe(WEEK1).fit(X_fit, y_fit)
point = odds_ratios(full)
rng = np.random.default_rng(SEED)
boot = []
for _ in range(300):                                           # refit on 300 resamples of the fitting people
    i = rng.integers(0, len(X_fit), len(X_fit))
    boot.append(odds_ratios(recipe(WEEK1).fit(X_fit.iloc[i], y_fit.iloc[i])))
low, high = pd.concat(boot, axis=1).quantile(0.025, axis=1), pd.concat(boot, axis=1).quantile(0.975, axis=1)
show = ["channel_paid_social", "channel_referral", "expenses_7d", "active_days_7d", "region_europe"]
print(pd.DataFrame({"odds ratio": point, "low": low, "high": high}).loc[show].round(3).to_string())

## Unit 3: Can the chances be read as chances?

**Predict first.** In the bin from 0.60 to 0.75, will the share of people who really churned be clearly below the mean chance the model gave them, near it, or clearly above it?

In [ ]:
p_val = full.predict_proba(X_val)[:, 1]
bins = pd.cut(p_val, [0, 0.3, 0.45, 0.6, 0.75, 1.0])
check_table = pd.DataFrame({"chance": p_val, "churned": y_val.to_numpy(), "bin": bins})
print(check_table.groupby("bin", observed=True).agg(people=("churned", "size"), mean_chance=("chance", "mean"), share_churned=("churned", "mean")).round(3).to_string())

## Unit 4: A threshold from error costs, chosen on the validation part

**Predict first.** With these costs, will the rule beat both "message nobody" and "message everybody" on the test part, and about what share of people will it flag: a quarter, a half or most?

In [ ]:
FP, FN = 2.0, 1.0                  # assumptions of this example: a wasted offer costs $2; a leaver left without an offer forgoes $1
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)


def cost_per_1000(p, y, t):
    """Cost of flagging everyone whose chance is at least t, for every 1,000 people."""
    flag = p >= t
    return 1000 * (FP * (flag & (y == 0)).sum() + FN * (~flag & (y == 1)).sum()) / len(y)


yv, yt = y_val.to_numpy(), y_test.to_numpy()
best = float(grid[int(np.argmin([cost_per_1000(p_val, yv, t) for t in grid]))])       # chosen on the validation part
print(round(FP / (FP + FN), 3), best)
p_test = full.predict_proba(X_test)[:, 1]                                              # the test part is used once, here
flag = p_test >= best
print(round(cost_per_1000(p_test, yt, best)), round(cost_per_1000(p_test, yt, 0.0)), round(cost_per_1000(p_test, yt, 1.01)))     # model, everybody, nobody
print(round(float(flag.mean()), 3), round(float(precision_score(yt, flag)), 3), round(float(recall_score(yt, flag)), 3))
rng = np.random.default_rng(SEED)
spread = []
for _ in range(300):                                           # how far does the cheapest threshold move when the validation people are resampled?
    i = rng.integers(0, len(yv), len(yv))
    spread.append(float(grid[int(np.argmin([cost_per_1000(p_val[i], yv[i], t) for t in grid]))]))
print(np.percentile(spread, [2.5, 50, 97.5]))

## Finish it

### Exercise 1: Finish it

**Task:** **How much of paid social's odds ratio is activity?** The starter reads the odds ratio of paid social in the model with the week-1 columns. Fit the same recipe with no number columns (the signup columns only), read the same odds ratio, and put the list [odds ratio without the week-1 columns, odds ratio with them] in `answer_1`, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of two numbers, the first larger than the second.

In [ ]:
with_week1 = odds_ratios(recipe(WEEK1).fit(X_fit, y_fit))["channel_paid_social"]      # given: the odds ratio of paid social in the model with the week-1 columns
# fit the same recipe with no number columns (the signup columns only) and read the same odds ratio
answer_1 = None  # replace None with [odds ratio without the week-1 columns, odds ratio with them], each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [4.283, 3.231], traps=[([3.231, 3.231], "The first number is the model without the week-1 columns: call recipe with an empty list.")], tol=5e-4)

<details>
<summary>Hint</summary>

`recipe([])` builds the model with no number columns. Read the same entry of `odds_ratios(...)`, and put the two numbers in the order the task gives.

</details>

<details>
<summary>Half-solution</summary>

```python
without_week1 = odds_ratios(recipe(...).fit(X_fit, y_fit))["channel_paid_social"]
answer_1 = [round(float(without_week1), 3), round(float(with_week1), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
without_week1 = odds_ratios(recipe([]).fit(X_fit, y_fit))["channel_paid_social"]
answer_1 = [round(float(without_week1), 3), round(float(with_week1), 3)]
```

Without the week-1 columns paid social has 4.283 times the odds of churning of organic search, and with them 3.231. Part of the raw gap is the activity of the first week; the rest is the channel itself. The odds ratio of a column changes with the columns beside it.

</details>

## On your own

### Exercise 2: On your own

**Task:** **A threshold for other costs.** Before you start, say in one sentence what the result looks like: a threshold well under 0.5, and a cost of a few hundred dollars. Now a wasted offer costs $1 and a leaver left without one forgoes $3. Use the validation chances `p_val` and the test chances `p_test` of unit 4 and its `grid`: find the cheapest threshold on the validation part, and put the list [that threshold, its cost for every 1,000 test people, rounded to a whole dollar] in `answer_2`.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.27, 441], traps=[([0.65, 464], "That uses the costs of unit 4, $2 and $1. Use $1 for a wasted offer and $3 for a leaver left without one.")], tol=5e-3)

<details>
<summary>Hint</summary>

Copy `cost_per_1000` with the new costs in it (give them names of their own, so that unit 4 still works), take the `argmin` over `grid` on the validation part, and score that threshold on the test part.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A threshold near the formula's 0.25, and a cost a little under half of the cost of messaging nobody."*

```python
best2 = float(grid[int(np.argmin([cost2(p_val, yv, t) for t in grid]))])
answer_2 = [best2, round(float(cost2(p_test, yt, best2)))]
```

</details>

<details>
<summary>Solution</summary>

```python
FP2, FN2 = 1.0, 3.0
def cost2(p, y, t):
    flag = p >= t
    return 1000 * (FP2 * (flag & (y == 0)).sum() + FN2 * (~flag & (y == 1)).sum()) / len(y)
best2 = float(grid[int(np.argmin([cost2(p_val, yv, t) for t in grid]))])
answer_2 = [best2, round(float(cost2(p_test, yt, best2)))]
```

The cheapest threshold on the validation part is 0.27, close to the formula's 0.25, and it costs $441 for every 1,000 test people. When a missed leaver costs three times a wasted offer, the rule flags more people: the costs move the threshold, and the model stays the same.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** The classes are 54 to 46, and a colleague "balances" them with `class_weight="balanced"` before using the threshold 0.667 on the chances. Fit the plain model instead and put in `answer_3` the number of test people it flags at 0.667.

**Expected:** `answer_3` is a whole number.

In [ ]:
weighted = recipe(WEEK1)
weighted.steps[-1] = ("logisticregression", LogisticRegression(max_iter=1000, class_weight="balanced"))      # colleague: "the classes are not 50 to 50, so I balance them"
weighted.fit(X_fit, y_fit)
answer_3 = int((weighted.predict_proba(X_test)[:, 1] >= 0.667).sum())          # colleague: how many test people get the offer at the threshold of 0.667
# change the model so that its chances can be read as chances, then count again

In [ ]:
_ = check("Exercise 3", "answer_3", 1003, traps={974: "That is the weighted model, whose chances are not chances any more. Fit the plain model."})

<details>
<summary>Hint</summary>

`class_weight="balanced"` changes how much each class counts when the model is fitted, and with it the chances. A threshold from the costs reads the chances as chances.

</details>

<details>
<summary>Half-solution</summary>

```python
plain = recipe(WEEK1).fit(X_fit, y_fit)
answer_3 = int((plain.predict_proba(...)[:, 1] >= 0.667).sum())
```

</details>

<details>
<summary>Solution</summary>

```python
plain = recipe(WEEK1).fit(X_fit, y_fit)
answer_3 = int((plain.predict_proba(X_test)[:, 1] >= 0.667).sum())
```

The plain model flags 1,003 of the 4,454 test people at 0.667; the weighted one flags 974. scikit-learn's "balanced" mode sets each class's weight to `n_samples / (n_classes * np.bincount(y))`, so the leavers count a little less and the stayers a little more, and every chance moves: the average chance on the test part is 0.508 for the weighted model and 0.547 for the plain one, against a real share of 0.544. The classes were never a problem: a 54 to 46 split needs no help, and the costs, not the weights, carry the asymmetry.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")